# 2. Limpieza, Unión y Transformación de datos

En este notebook, tomaremos los datos brutos combinados, los limpiaremos, los uniremos con la información de las estaciones y crearemos nuevas características para el análisis.

In [2]:
# Importar librerías y carga de datos
import pandas as pd
import numpy as np

# Cargar datos de viajes (con Dtype para evitar warnings) 
tipos_viajes = {
    'ride_id': str,
    'start_station_id': str,
    'end_station_id': str
}
df_trips = pd.read_csv(
    '../data/processed/trips_raw_combined.csv', 
    dtype=tipos_viajes
)

# Cargar datos de estaciones desde la fuente oficial 
url_estaciones = 'https://gbfs.citibikenyc.com/gbfs/en/station_information.json'
df_stations_raw = pd.read_json(url_estaciones)
df_stations = pd.json_normalize(df_stations_raw['data']['stations'])

print("Paso 1: Datos cargados.")

Paso 1: Datos cargados.


## 2.1 Preparación y Unión de datos

Primero, preparamos el DataFrame de estaciones y nos aseguramos de que los tipos de datos de las columnas clave coincidan antes de realizar la unión.

***Se detectó que los sistemas de ID no eran compatibles, por lo que se procedió a unir los datos por el nombre de la estación***

In [3]:
# Se prepara df_stations: se selecciona y renombra columnas
df_stations = df_stations[['station_id', 'name', 'lat', 'lon']].rename(columns={'name': 'station_name', 'lat': 'latitude', 'lon': 'longitude'})

# Se prepara las columnas clave para la unión por NOMBRE
df_trips['start_station_name'] = df_trips['start_station_name'].astype(str).str.strip()
df_trips['end_station_name'] = df_trips['end_station_name'].astype(str).str.strip()
df_stations['station_name'] = df_stations['station_name'].astype(str).str.strip()

print("Paso 2: Datos preparados para la unión.")

# Unión de datos
df_merged = pd.merge(df_trips, df_stations, left_on='start_station_name', right_on='station_name', how='left')
df_final = pd.merge(df_merged, df_stations, left_on='end_station_name', right_on='station_name', how='left',suffixes=('_start', '_end'))

print("Paso 3: Unión completada.")

Paso 2: Datos preparados para la unión.
Paso 3: Unión completada.


## 2.2 Transformación, Limpieza final y Guardado

Creamos las características adicionales, aplicamos los filtros (outliers, nulos, duplicados), seleccionamos las columnas finales y guardamos el resultado.

In [4]:
# Convertimos fechas
df_final['started_at'] = pd.to_datetime(df_final['started_at'])
df_final['ended_at'] = pd.to_datetime(df_final['ended_at'])

# Creamos nuevas columnas
# 1. Calcular la duración del viaje en minutos
df_final['trip_duration_minutes'] = (df_final['ended_at'] - df_final['started_at']).dt.total_seconds() / 60
# 2. Extraer la hora de inicio del viaje (para ver patrones horarios)
df_final['start_hour'] = df_final['started_at'].dt.hour
# 3. Extraer el día de la semana (para ver patrones semanales)
df_final['day_of_week'] = df_final['started_at'].dt.day_name()
# 4. Extraer el mes (para ver patrones estacionales)
df_final['month'] = df_final['started_at'].dt.month_name()

# Limpieza
# Filtramos outliers, nulos y duplicados
df_final = df_final[(df_final['trip_duration_minutes'] > 1) & (df_final['trip_duration_minutes'] <= 1440)]
columnas_criticas = ['station_name_start', 'latitude_start', 'longitude_start', 'station_name_end', 'latitude_end', 'longitude_end']
df_final.dropna(subset=columnas_criticas, inplace=True)
df_final.drop_duplicates(subset=['ride_id'], inplace=True)

# Seleccionamos columnas finales para evitar información no importante para el análisis
columnas_para_analisis = [
    'ride_id', 'rideable_type', 'started_at', 'ended_at', 'member_casual', 
    'trip_duration_minutes', 'start_hour', 'day_of_week', 'month', 
    'station_name_start', 'latitude_start', 'longitude_start', 
    'station_name_end', 'latitude_end', 'longitude_end'
]
df_analisis = df_final[columnas_para_analisis].copy()
for col in ['member_casual', 'day_of_week', 'month', 'rideable_type']:
    df_analisis[col] = df_analisis[col].astype('category')

# Comprobación final
print("DataFrame final")
df_analisis.info()

# Guardado del archivo final
df_analisis.to_csv('../data/processed/trips_final_for_analysis.csv', index=False)

print("¡DataFrame limpio y guardado correctamente!")

DataFrame final
<class 'pandas.core.frame.DataFrame'>
Index: 3131873 entries, 0 to 3193373
Data columns (total 15 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   ride_id                object        
 1   rideable_type          category      
 2   started_at             datetime64[ns]
 3   ended_at               datetime64[ns]
 4   member_casual          category      
 5   trip_duration_minutes  float64       
 6   start_hour             int32         
 7   day_of_week            category      
 8   month                  category      
 9   station_name_start     object        
 10  latitude_start         float64       
 11  longitude_start        float64       
 12  station_name_end       object        
 13  latitude_end           float64       
 14  longitude_end          float64       
dtypes: category(4), datetime64[ns](2), float64(5), int32(1), object(3)
memory usage: 286.7+ MB
¡DataFrame limpio y guardado correctamente!


In [5]:
# Mostramos las primeras 10 filas para comprobar el trabajo realizado
df_analisis.head(10)

,ride_id,rideable_type,started_at,ended_at,member_casual,trip_duration_minutes,start_hour,day_of_week,month,station_name_start,latitude_start,longitude_start,station_name_end,latitude_end,longitude_end
0,0AE5FB7DFCB8E5BE,classic_bike,2025-03-11 17:33:37.322,2025-03-11 17:36:58.910,member,3.359800,17,Tuesday,March,Chauncey St & Stuyvesant Ave,40.680120,-73.931680,Lewis Ave & Decatur St,40.681460,-73.934903
1,0F8A4821A038CDBE,electric_bike,2025-03-04 07:48:17.001,2025-03-04 07:52:52.595,member,4.593233,7,Tuesday,March,Bergen St & Smith St,40.686744,-73.990632,Bergen St & Flatbush Ave,40.680945,-73.975673
2,17F184F80E088790,electric_bike,2025-03-01 20:28:36.021,2025-03-01 20:36:41.340,member,8.088650,20,Saturday,March,Bergen St & Smith St,40.686744,-73.990632,Johnson St & Gold St,40.694749,-73.983625
3,241B925B376A39B3,electric_bike,2025-03-05 09:25:53.437,2025-03-05 09:36:50.983,member,10.959100,9,Wednesday,March,Ave A & E 14 St,40.730311,-73.980472,W 20 St & 7 Ave,40.742388,-73.997262
4,2BCAB8EEAEC26BFB,electric_bike,2025-03-12 17:53:59.432,2025-03-12 17:57:01.529,member,3.034950,17,Wednesday,March,W 4 St & 7 Ave S,40.734011,-74.002939,King St & Varick St,40.727897,-74.005363
5,F4E67646FE9A6DF0,electric_bike,2025-03-05 01:25:13.277,2025-03-05 01:32:53.453,member,7.669600,1,Wednesday,March,Broadway & Kosciuszko St,40.693290,-73.928520,Irving Ave & DeKalb Ave,40.702700,-73.920950
6,E4B67D6633ADA7A6,electric_bike,2025-03-09 12:26:45.340,2025-03-09 12:35:58.411,member,9.217850,12,Sunday,March,Pacific St & Nostrand Ave,40.677600,-73.949630,Wyckoff St & 3 Ave,40.682755,-73.982586
7,815C0ACA4BE047FB,classic_bike,2025-03-12 09:43:06.615,2025-03-12 10:23:12.415,member,40.096667,9,Wednesday,March,Cedar St & Evergreen Ave,40.696710,-73.928070,Elmhurst Ave & Roosevelt Ave,40.748640,-73.876190
8,CAA40A0B9521B20B,electric_bike,2025-03-01 15:09:47.305,2025-03-01 15:11:35.483,member,1.802967,15,Saturday,March,E 33 St & 1 Ave,40.743227,-73.974498,E 33 St & 1 Ave,40.743227,-73.974498
9,8A848776E2B39712,classic_bike,2025-03-10 07:07:13.338,2025-03-10 08:10:30.611,member,63.287883,7,Monday,March,E 33 St & 1 Ave,40.743227,-73.974498,E 33 St & 1 Ave,40.743227,-73.974498
